# Baseline model — Arequipa price prediction

Exploratory only — same role as the previous two notebooks: figuring things out interactively and documenting decisions with real evidence. The repeatable version moves into `ml/training/train.py`.

Goal: confirm the problem is viable (the model learns something reasonable) before investing time in infrastructure (Feast, Docker, MLflow). No infra used here — trains directly on `data/processed/features.parquet`.

Open questions to resolve in this notebook:
1. Categorical encoding for `district`, `property_type`, `operation_type`.
2. One model with `operation_type` as a feature, or two separate models (Venta/Alquiler)?
3. Train on `price_usd` directly, or `log(price_usd)`?
4. Train/test split strategy and evaluation metrics.

**Library: XGBoost**, decided without needing an empirical comparison — both XGBoost and LightGBM would give similar baseline quality on a dataset this size (6,825 rows), so the choice comes down to two concrete factors instead: XGBoost's ONNX export path is more mature (needed later for the Node.js inference API), and it has native categorical support (`enable_categorical=True`), which feeds directly into question 1 above.

## Categorical encoding: native categorical vs one-hot

`district`, `property_type`, `operation_type` need encoding. Two options: XGBoost's native categorical support (`enable_categorical=True`), or the usual `pd.get_dummies` one-hot. Comparing both directly rather than assuming.

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

features = pd.read_parquet("../data/processed/features.parquet")
cat_cols = ["district", "property_type", "operation_type"]
X = features.drop(columns=["id", "price_usd"])
y = features["price_usd"]

# Quick 80/20 split just to compare encodings on equal footing — the real
# split strategy is a separate, later decision.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for c in cat_cols:
    unseen = set(X_test[c].unique()) - set(X_train[c].unique())
    if unseen:
        print(f"{c}: category unseen in train but present in test: {unseen}")

district: category unseen in train but present in test: {'Camana'}


Real, not just a random-split edge case to shrug off: `Camana` (a small district) landed entirely in the test split, so it's genuinely unseen at "training time" here — the same thing will happen in production whenever a district shows up that had too few historical listings to land in a given training run, or a genuinely new area gets added. This needs explicit handling either way, not just for this comparison: map unseen categories to `NaN`, which XGBoost treats as missing and routes natively during tree splits — no crash, no silent wrong-category coercion.

In [2]:
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for c in cat_cols:
    X_train_cat[c] = X_train_cat[c].astype("category")
    categories = X_train_cat[c].cat.categories
    X_test_cat[c] = X_test_cat[c].where(X_test_cat[c].isin(categories))
    X_test_cat[c] = X_test_cat[c].astype(pd.CategoricalDtype(categories=categories))

model_cat = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
model_cat.fit(X_train_cat, y_train)
pred_cat = model_cat.predict(X_test_cat)
print("Native categorical -> R2:", round(r2_score(y_test, pred_cat), 4), " MAE:", round(mean_absolute_error(y_test, pred_cat), 2))

X_oh = pd.get_dummies(X, columns=cat_cols)
X_train_oh, X_test_oh, _, _ = train_test_split(X_oh, y, test_size=0.2, random_state=42)
model_oh = xgb.XGBRegressor(random_state=42)
model_oh.fit(X_train_oh, y_train)
pred_oh = model_oh.predict(X_test_oh)
print("One-hot -> R2:", round(r2_score(y_test, pred_oh), 4), " MAE:", round(mean_absolute_error(y_test, pred_oh), 2))
print()
print("n features -> native:", X_train_cat.shape[1], " one-hot:", X_train_oh.shape[1])

Native categorical -> R2: 0.8519  MAE: 32901.25
One-hot -> R2: 0.819  MAE: 33890.56

n features -> native: 5  one-hot: 34


**Decision: native categorical encoding**, not one-hot. Native wins on both metrics (R² 0.852 vs 0.819, lower MAE) while using 5 columns instead of 34 — one-hot's dimensionality blow-up (mostly from `district`'s ~20 values) doesn't buy anything here, tree-based splits handle categories directly and more efficiently. Unseen categories (like `Camana` above) map to `NaN`, which XGBoost routes as missing during splits — same handling needed in `ml/training/train.py` and later at inference time.

## One model with `operation_type` as a feature, or two separate models?

`operation_type` has already shown a ~150x price scale gap between `Venta` and `Alquiler` repeatedly (cleaning notebook, feature EDA notebook). Testing directly: train one unified model (using the encoding just decided above) and check its error **per group**, versus training two separate models.

In [3]:
def to_cat(train_df, test_df, cols):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for c in cols:
        train_df[c] = train_df[c].astype("category")
        categories = train_df[c].cat.categories
        test_df[c] = test_df[c].where(test_df[c].isin(categories))
        test_df[c] = test_df[c].astype(pd.CategoricalDtype(categories=categories))
    return train_df, test_df


print("=== Unified model (operation_type as feature), evaluated per group ===")
for op in ["Venta", "Alquiler"]:
    mask = X_test["operation_type"] == op
    r2 = r2_score(y_test[mask], pred_cat[mask])
    mae = mean_absolute_error(y_test[mask], pred_cat[mask])
    print(f"{op}: n={mask.sum()}  R2={r2:.4f}  MAE={mae:.2f}")

=== Unified model (operation_type as feature), evaluated per group ===
Venta: n=958  R2=0.8423  MAE=42931.39
Alquiler: n=407  R2=-2593.6464  MAE=9292.22


**Catastrophic, not just "worse."** The unified model's R² on `Alquiler` is wildly negative — far worse than just predicting the mean rent. Gradient boosting minimizes squared error in raw dollars, and `Venta`'s errors are inherently thousands of times larger in magnitude than `Alquiler`'s, so the training loss is completely dominated by `Venta` — the model has essentially no incentive to fit `Alquiler` at all.

Before concluding "always separate models," checking whether this is actually a raw-target-scale artifact that log-transforming would fix — that's literally the next decision on the list, so worth resolving the overlap now rather than assuming.

In [4]:
import numpy as np

log_y_train = np.log(y_train)
model_log_unified = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
model_log_unified.fit(X_train_cat, log_y_train)
pred_log_unified = np.exp(model_log_unified.predict(X_test_cat))

print("=== Unified model, trained on log(price_usd), evaluated per group (back-transformed) ===")
for op in ["Venta", "Alquiler"]:
    mask = X_test["operation_type"] == op
    r2 = r2_score(y_test[mask], pred_log_unified[mask])
    mae = mean_absolute_error(y_test[mask], pred_log_unified[mask])
    print(f"{op}: n={mask.sum()}  R2={r2:.4f}  MAE={mae:.2f}")

=== Unified model, trained on log(price_usd), evaluated per group (back-transformed) ===


Venta: n=958  R2=0.7852  MAE=43705.43
Alquiler: n=407  R2=-0.0753  MAE=273.71


Log-transforming mostly fixes it (R² goes from -2593 to -0.08) — confirms the catastrophe above was largely a raw-scale training-loss artifact, not proof that a single model can never work. But `Alquiler`'s R² is still negative — the unified log-model is still no better than predicting the mean rent. Now comparing against two separate models, trained and evaluated independently on each group (same train/test split, `operation_type` dropped since it's constant within each).

In [5]:
def split_operation(df, operation_type, feature_cols, target_col="price_usd",
                     cat_cols=("district", "property_type"), test_size=0.2, random_state=42):
    """Filter to one operation_type, then split independently — the two
    models don't interact, so there's no reason to share a split across
    them. Matches ml/training/train.py's train_operation_model exactly,
    rather than reusing the global split from the unified-model comparison
    above (fine for that comparison, but not the right mechanism once each
    operation_type gets its own model).
    """
    sub = df[df["operation_type"] == operation_type]
    X = sub[feature_cols]
    y = sub[target_col]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=random_state)
    Xtr, Xte = to_cat(Xtr, Xte, list(cat_cols))
    return Xtr, Xte, ytr, yte


sub_cols = ["district", "surface", "property_type", "district_avg_price_per_m2"]

In [6]:
print("=== Separate models (one per operation_type), raw price_usd ===")
for op in ["Venta", "Alquiler"]:
    Xtr, Xte, ytr, yte = split_operation(features, op, sub_cols)
    m = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)

    r2 = r2_score(yte, pred)
    mae = mean_absolute_error(yte, pred)
    print(f"{op}: n_train={len(Xtr)} n_test={len(Xte)}  R2={r2:.4f}  MAE={mae:.2f}")

=== Separate models (one per operation_type), raw price_usd ===


Venta: n_train=3832 n_test=958  R2=0.8328  MAE=41913.85


Alquiler: n_train=1628 n_test=407  R2=0.3968  MAE=320.74


**Decision: two separate models (Venta, Alquiler), not one unified model.** Separate models (even before picking a target transform for them) already beat the unified log-model on `Venta` (R² 0.83 vs 0.79) and clearly beat it on `Alquiler` (R² 0.40 vs -0.08) — a large, real gap. Makes sense beyond the numbers too: a sale price and a monthly rent are different economic quantities (one-time payment vs. recurring), not just the same quantity at different scale — asking one model to learn both from a single `operation_type` flag is a harder problem than just training two models on data that's already naturally separable with zero cost (nothing here requires them to share parameters). `ml/training/train.py` trains and saves one model per `operation_type`.

## `price_usd` direct or `log(price_usd)` — decided per model

Since Venta and Alquiler are now separate models, this needs its own answer for each — no reason to assume the same target transform is best for both. Training both variants for each model, same split as above.

In [7]:
results = {}
for op in ["Venta", "Alquiler"]:
    Xtr, Xte, ytr, yte = split_operation(features, op, sub_cols)

    m_raw = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
    m_raw.fit(Xtr, ytr)
    pred_raw = m_raw.predict(Xte)

    m_log = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
    m_log.fit(Xtr, np.log(ytr))
    pred_log = np.exp(m_log.predict(Xte))

    results[op] = dict(
        raw_r2=r2_score(yte, pred_raw), raw_mae=mean_absolute_error(yte, pred_raw),
        log_r2=r2_score(yte, pred_log), log_mae=mean_absolute_error(yte, pred_log),
    )
    print(f"--- {op} ---")
    print(f"  raw price_usd:      R2={results[op]['raw_r2']:.4f}  MAE={results[op]['raw_mae']:.2f}")
    print(f"  log(price_usd):     R2={results[op]['log_r2']:.4f}  MAE={results[op]['log_mae']:.2f}")

--- Venta ---
  raw price_usd:      R2=0.8328  MAE=41913.85
  log(price_usd):     R2=0.8730  MAE=32356.04


--- Alquiler ---
  raw price_usd:      R2=0.3968  MAE=320.74
  log(price_usd):     R2=0.6221  MAE=254.78


**Decision: `log(price_usd)` for both models** — no split needed, log wins clearly on both R² and MAE for each. Biggest effect on `Alquiler`: R² 0.40 → 0.62, a bigger jump than `Venta`'s 0.83 → 0.87. Makes sense: real estate prices are right-skewed (a long tail of expensive properties), and log-transform is the standard fix for that — same reasoning already used for the surface/price correlation check in the cleaning notebook. `ml/training/train.py` trains both models on `log(price_usd)` and exponentiates predictions back to dollars.

## Train/test split strategy

Two options worth considering: stratify by `operation_type` (moot now — each model already gets only its own operation type, so there's nothing left to stratify there), or stratify by `district` (to guarantee every district appears in both train and test, avoiding the `Camana`-style unseen-category case from earlier). Checking whether district-stratification is even possible before assuming it's the better option.

In [8]:
for op in ["Venta", "Alquiler"]:
    sub = features[features["operation_type"] == op]
    counts = sub["district"].value_counts()
    print(f"--- {op}: {len(sub)} rows, {counts.shape[0]} districts ---")
    print("districts with 1 row (can't be stratified — needs >=2 per group):", (counts == 1).sum())
    print(counts[counts < 5].to_dict())
    print()

--- Venta: 4790 rows, 22 districts ---
districts with 1 row (can't be stratified — needs >=2 per group): 1
{'Mollendo': 1}

--- Alquiler: 2035 rows, 16 districts ---
districts with 1 row (can't be stratified — needs >=2 per group): 2
{'Yura': 2, 'Camana': 1, 'Characato': 1}



**Decision: plain random split** (`test_size=0.2`, fixed `random_state=42` for reproducibility), no stratification by district. `Mollendo` (Venta) and `Camana`/`Characato` (Alquiler) have exactly 1 listing each — stratification is mechanically impossible for them (scikit-learn requires at least 2 members per stratum), so a "stratify by district" strategy would need special-casing for tiny districts anyway. Not worth the complexity for a baseline: the unseen-category handling already built (map to `NaN`, XGBoost routes as missing) means an unlucky split like `Camana`'s doesn't break anything, just costs a little accuracy on that one row — an acceptable, realistic trade-off (production will see genuinely new districts too, this isn't just a split artifact to engineer away).

## Evaluation metrics

All in original dollar scale (back-transformed from the log model), not log-scale — log-RMSE isn't interpretable to a non-technical reader, and the product's whole pitch is "is this price reasonable," which is fundamentally a percentage question. Also comparing against a trivial mean-prediction baseline, since R²/MAE alone don't establish whether the model is actually learning anything.

In [9]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

for op in ["Venta", "Alquiler"]:
    Xtr, Xte, ytr, yte = split_operation(features, op, sub_cols)

    m = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
    m.fit(Xtr, np.log(ytr))
    pred = np.exp(m.predict(Xte))

    rmse = mean_squared_error(yte, pred) ** 0.5
    mae = mean_absolute_error(yte, pred)
    mape = mean_absolute_percentage_error(yte, pred) * 100
    r2 = r2_score(yte, pred)
    print(f"--- {op} (n_test={len(Xte)}) ---")
    print(f"  R2={r2:.4f}  RMSE={rmse:,.2f}  MAE={mae:,.2f}  MAPE={mape:.1f}%")

    naive_pred = np.full_like(yte, ytr.mean(), dtype=float)
    naive_r2 = r2_score(yte, naive_pred)
    naive_mape = mean_absolute_percentage_error(yte, naive_pred) * 100
    print(f"  [trivial mean baseline: R2={naive_r2:.4f}  MAPE={naive_mape:.1f}%]")

--- Venta (n_test=958) ---


  R2=0.8730  RMSE=124,394.26  MAE=32,356.04  MAPE=15.1%
  [trivial mean baseline: R2=-0.0002  MAPE=119.3%]
--- Alquiler (n_test=407) ---
  R2=0.6221  RMSE=977.78  MAE=254.78  MAPE=14.8%
  [trivial mean baseline: R2=-0.0002  MAPE=128.8%]


**Decision: track R², RMSE, MAE, and MAPE, all in dollar scale.** MAPE is the headline metric — most directly answers "is this price reasonable" (15.1% off on Venta, 14.8% on Alquiler), and is comparable across the two models despite their completely different price scales, unlike RMSE/MAE in raw dollars. RMSE/MAE stay in the report as standard, complementary regression metrics (RMSE penalizes large misses more; useful for catching outlier-driven error that MAPE alone could hide). Both models clear the trivial mean-baseline by a wide margin (baseline MAPE 119-129%), which doubles as evidence the model is actually learning something, not just fitting noise. `ml/training/train.py` prints all four for both models.

## Final baseline: train and record metrics

Every decision above (native categorical encoding, two separate models, log target, random split, R²/RMSE/MAE/MAPE) applied together, one last time, as a single clean run — this is the baseline result that gets recorded.

In [10]:
baseline_results = []
trained_models = {}

for op in ["Venta", "Alquiler"]:
    Xtr, Xte, ytr, yte = split_operation(features, op, sub_cols)

    model = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42)
    model.fit(Xtr, np.log(ytr))
    pred = np.exp(model.predict(Xte))
    trained_models[op] = model

    naive_pred = np.full_like(yte, ytr.mean(), dtype=float)
    baseline_results.append(dict(
        operation_type=op,
        n_train=len(Xtr), n_test=len(Xte),
        r2=r2_score(yte, pred),
        rmse=mean_squared_error(yte, pred) ** 0.5,
        mae=mean_absolute_error(yte, pred),
        mape_pct=mean_absolute_percentage_error(yte, pred) * 100,
        naive_mean_mape_pct=mean_absolute_percentage_error(yte, naive_pred) * 100,
    ))

baseline_df = pd.DataFrame(baseline_results).set_index("operation_type")
baseline_df

,n_train,n_test,r2,rmse,mae,mape_pct,naive_mean_mape_pct
operation_type,,,,,,,
Venta,3832,958,0.873027,124394.257615,32356.039617,15.076838,119.330529
Alquiler,1628,407,0.622113,977.777443,254.778425,14.835761,128.797697


## Is this actually viable? Confirming before moving to infrastructure

The whole point of this baseline is to confirm the problem is learnable *before* spending time on Docker/Postgres/Feast. "Better than predicting the mean" is the bar — checking both models clear it clearly, not just marginally.

In [11]:
verdict = baseline_df[["r2", "mape_pct", "naive_mean_mape_pct"]].copy()
verdict["mape_improvement_x"] = verdict["naive_mean_mape_pct"] / verdict["mape_pct"]
verdict["beats_trivial_baseline"] = verdict["r2"] > 0
verdict

,r2,mape_pct,naive_mean_mape_pct,mape_improvement_x,beats_trivial_baseline
operation_type,,,,,
Venta,0.873027,15.076838,119.330529,7.914825,True
Alquiler,0.622113,14.835761,128.797697,8.681570,True


**Confirmed: the problem is viable.** Both models clear the trivial mean-baseline decisively, not marginally — `Venta`'s MAPE is ~7.9x lower than the naive baseline, `Alquiler`'s is ~8.7x lower, and both R² are solidly positive. Safe to move on to infrastructure — the model learns real signal from district, surface, and property type, not just noise.

## Encapsulated in `ml/training/train.py`

`train.py`'s `train_operation_model` uses the exact same split mechanism as `split_operation` above (filter to one `operation_type`, then split independently, `test_size=0.2`, `random_state=42`) — so its printed metrics match this notebook's exactly, not just approximately. Running `python3 ml/training/train.py` reproduces `R2=0.8730`/`MAPE=15.1%` for Venta and `R2=0.6221`/`MAPE=14.8%` for Alquiler.